# Saúde em Foco — Análise de dados do ABC Paulista

**FIAP | Global Solution 2026.2**  
**Objetivo:** integrar dados populacionais do IBGE (Censo 2022) e registros de estabelecimentos do CNES para os sete municípios do ABC Paulista.

> **Atenção metodológica:** a coluna `CO_IBGE` do arquivo CNES contém o código municipal de seis dígitos. A tabela do IBGE utiliza o código completo de sete dígitos; por isso, o relacionamento é feito pelos seis primeiros dígitos. A contagem inicial de registros não deve ser interpretada automaticamente como quantidade de unidades públicas, ativas ou com atendimento SUS.

## 1. Preparação do ambiente e bibliotecas

Importa as bibliotecas e cria as pastas usadas para armazenar dados brutos e tratados.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import json
import re
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

PASTA_DADOS = Path("dados")
PASTA_BRUTOS = PASTA_DADOS / "brutos"
PASTA_TRATADOS = PASTA_DADOS / "tratados"

PASTA_BRUTOS.mkdir(parents=True, exist_ok=True)
PASTA_TRATADOS.mkdir(parents=True, exist_ok=True)

## 2. Definição dos municípios do ABC Paulista

Cria a tabela de referência com os sete municípios e seus códigos IBGE.

In [2]:
municipios_abc = pd.DataFrame({
    "codigo_ibge": [
        "3513801", "3529401", "3543303", "3544103",
        "3547809", "3548708", "3548807"
    ],
    "municipio": [
        "Diadema", "Mauá", "Ribeirão Pires", "Rio Grande da Serra",
        "Santo André", "São Bernardo do Campo", "São Caetano do Sul"
    ],
    "uf": ["SP"] * 7
})

municipios_abc

,codigo_ibge,municipio,uf
0,3513801,Diadema,SP
1,3529401,Mauá,SP
2,3543303,Ribeirão Pires,SP
3,3544103,Rio Grande da Serra,SP
4,3547809,Santo André,SP
5,3548708,São Bernardo do Campo,SP
6,3548807,São Caetano do Sul,SP


## 3. Consulta da população no IBGE/SIDRA

Obtém a população residente do Censo 2022 para os sete municípios.

In [3]:
arquivo_populacao_local = Path("populacao_abc_censo_2022.csv")
arquivo_populacao_tratada = PASTA_TRATADOS / "populacao_abc_censo_2022.csv"

if arquivo_populacao_local.exists():
    populacao_ibge = pd.read_csv(arquivo_populacao_local, dtype={"codigo_ibge": str})
    print("População carregada do CSV enviado junto ao notebook.")
elif arquivo_populacao_tratada.exists():
    populacao_ibge = pd.read_csv(arquivo_populacao_tratada, dtype={"codigo_ibge": str})
    print("População carregada da pasta de dados tratados.")
else:
    url_sidra = (
        "https://apisidra.ibge.gov.br/values/t/4709/"
        "n6/3513801,3529401,3543303,3544103,3547809,3548708,3548807/"
        "v/93/p/2022"
    )
    resposta = requests.get(url_sidra, timeout=60)
    resposta.raise_for_status()
    dados_sidra = resposta.json()
    populacao_ibge = pd.DataFrame(dados_sidra[1:])

print("Registros populacionais obtidos:", len(populacao_ibge))
display(populacao_ibge.head())

População carregada da pasta de dados tratados.
Registros populacionais obtidos: 7


,codigo_ibge,municipio_ibge,populacao_2022
0,3513801,Diadema (SP),393237
1,3529401,Mauá (SP),418261
2,3543303,Ribeirão Pires (SP),115559
3,3544103,Rio Grande da Serra (SP),44170
4,3547809,Santo André (SP),748919


## 4. Tratamento dos dados populacionais

Padroniza os campos e mantém as colunas necessárias para a integração.

In [4]:
populacao_ibge = populacao_ibge.rename(columns={
    "D1C": "codigo_ibge",
    "D1N": "municipio_ibge",
    "V": "populacao_2022"
})

populacao_ibge["codigo_ibge"] = populacao_ibge["codigo_ibge"].astype(str)
populacao_ibge["populacao_2022"] = pd.to_numeric(
    populacao_ibge["populacao_2022"], errors="coerce"
)

populacao_ibge = populacao_ibge[
    ["codigo_ibge", "municipio_ibge", "populacao_2022"]
].drop_duplicates()

populacao_ibge

,codigo_ibge,municipio_ibge,populacao_2022
0,3513801,Diadema (SP),393237
1,3529401,Mauá (SP),418261
2,3543303,Ribeirão Pires (SP),115559
3,3544103,Rio Grande da Serra (SP),44170
4,3547809,Santo André (SP),748919
5,3548708,São Bernardo do Campo (SP),810729
6,3548807,São Caetano do Sul (SP),165655


## 5. Validação da população municipal

Confere se os sete municípios foram retornados e exporta a base populacional.

In [5]:
# Conferência: esperamos exatamente os sete municípios.
conferencia_ibge = municipios_abc.merge(
    populacao_ibge,
    on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

display(conferencia_ibge)

assert len(conferencia_ibge) == 7, "A consulta não retornou os sete municípios."
assert conferencia_ibge["populacao_2022"].notna().all(), (
    "Há município sem população. Confira a resposta da API SIDRA."
)

populacao_ibge.to_csv(
    PASTA_TRATADOS / "populacao_abc_censo_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

,codigo_ibge,municipio,uf,municipio_ibge,populacao_2022
0,3513801,Diadema,SP,Diadema (SP),393237
1,3529401,Mauá,SP,Mauá (SP),418261
2,3543303,Ribeirão Pires,SP,Ribeirão Pires (SP),115559
3,3544103,Rio Grande da Serra,SP,Rio Grande da Serra (SP),44170
4,3547809,Santo André,SP,Santo André (SP),748919
5,3548708,São Bernardo do Campo,SP,São Bernardo do Campo (SP),810729
6,3548807,São Caetano do Sul,SP,São Caetano do Sul (SP),165655


## 6. Download do arquivo CNES

Baixa o arquivo compactado do CNES quando ele ainda não está disponível localmente.

In [6]:
url_cnes = (
    "https://s3.sa-east-1.amazonaws.com/"
    "ckan.saude.gov.br/CNES/cnes_estabelecimentos_json.zip"
)

# Se o ZIP enviado estiver na mesma pasta do notebook, utiliza esse arquivo.
arquivo_zip_local = Path("cnes_estabelecimentos_json.zip")
arquivo_zip_cnes = arquivo_zip_local if arquivo_zip_local.exists() else (PASTA_BRUTOS / "cnes_estabelecimentos_json.zip")

if not arquivo_zip_cnes.exists():
    with requests.get(url_cnes, stream=True, timeout=180) as resposta:
        resposta.raise_for_status()
        with open(arquivo_zip_cnes, "wb") as arquivo:
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                if bloco:
                    arquivo.write(bloco)

print("Arquivo CNES:", arquivo_zip_cnes.resolve())
print(f"Tamanho compactado: {arquivo_zip_cnes.stat().st_size / (1024**2):.1f} MB")

Arquivo CNES: C:\Users\Caue Almiron\Documents\global_fiap\dados\brutos\cnes_estabelecimentos_json.zip
Tamanho compactado: 65.4 MB


## 7. Inspeção do arquivo compactado CNES

Lista o JSON existente no ZIP sem carregar todo o arquivo na memória.

In [7]:
with ZipFile(arquivo_zip_cnes) as z:
    arquivos_json = [nome for nome in z.namelist() if nome.lower().endswith(".json")]
    print("Arquivos JSON encontrados:", arquivos_json)

if not arquivos_json:
    raise ValueError("Não foi encontrado arquivo JSON dentro do ZIP.")

nome_json = arquivos_json[0]
print("Arquivo que será processado:", nome_json)
print("Leitura em fluxo será usada para reduzir o consumo de memória.")

Arquivos JSON encontrados: ['cnes_estabelecimentos.json']
Arquivo que será processado: cnes_estabelecimentos.json
Leitura em fluxo será usada para reduzir o consumo de memória.


## 8. Extração e normalização dos registros CNES

Lê o JSON em fluxo e mantém apenas os registros dos municípios do ABC Paulista.

In [10]:
# Instale ijson uma única vez, se necessário: %pip install ijson
import ijson

# CO_IBGE no arquivo CNES possui seis dígitos; o código IBGE completo possui sete.
codigos_abc_6 = set(municipios_abc["codigo_ibge"].astype(str).str[:6])
registros_abc = []

with ZipFile(arquivo_zip_cnes) as z:
    with z.open(nome_json) as arquivo_json:
        for registro in ijson.items(arquivo_json, "item"):
            codigo = str(registro.get("CO_IBGE", "")).strip()
            if codigo in codigos_abc_6:
                registros_abc.append(registro)

cnes_abc = pd.DataFrame(registros_abc)
print("Dimensão da base filtrada:", cnes_abc.shape)
print("Colunas disponíveis:", cnes_abc.columns.tolist())
display(cnes_abc.head(3))

Dimensão da base filtrada: (6290, 36)
Colunas disponíveis: ['CO_CNES', 'CO_UNIDADE', 'CO_UF', 'CO_IBGE', 'NU_CNPJ_MANTENEDORA', 'NO_RAZAO_SOCIAL', 'NO_FANTASIA', 'CO_NATUREZA_ORGANIZACAO', 'DS_NATUREZA_ORGANIZACAO', 'TP_GESTAO', 'CO_NIVEL_HIERARQUIA', 'DS_NIVEL_HIERARQUIA', 'CO_ESFERA_ADMINISTRATIVA', 'DS_ESFERA_ADMINISTRATIVA', 'CO_ATIVIDADE', 'TP_UNIDADE', 'CO_CEP', 'NO_LOGRADOURO', 'NU_ENDERECO', 'NO_BAIRRO', 'NU_TELEFONE', 'NU_LATITUDE', 'NU_LONGITUDE', 'CO_TURNO_ATENDIMENTO', 'DS_TURNO_ATENDIMENTO', 'NU_CNPJ', 'NO_EMAIL', 'CO_NATUREZA_JUR', 'ST_CENTRO_CIRURGICO', 'ST_CENTRO_OBSTETRICO', 'ST_CENTRO_NEONATAL', 'ST_ATEND_HOSPITALAR', 'ST_SERVICO_APOIO', 'ST_ATEND_AMBULATORIAL', 'CO_MOTIVO_DESAB', 'CO_AMBULATORIAL_SUS']


,CO_CNES,CO_UNIDADE,CO_UF,CO_IBGE,NU_CNPJ_MANTENEDORA,NO_RAZAO_SOCIAL,NO_FANTASIA,CO_NATUREZA_ORGANIZACAO,DS_NATUREZA_ORGANIZACAO,TP_GESTAO,CO_NIVEL_HIERARQUIA,DS_NIVEL_HIERARQUIA,CO_ESFERA_ADMINISTRATIVA,DS_ESFERA_ADMINISTRATIVA,CO_ATIVIDADE,TP_UNIDADE,CO_CEP,NO_LOGRADOURO,NU_ENDERECO,NO_BAIRRO,NU_TELEFONE,NU_LATITUDE,NU_LONGITUDE,CO_TURNO_ATENDIMENTO,DS_TURNO_ATENDIMENTO,NU_CNPJ,NO_EMAIL,CO_NATUREZA_JUR,ST_CENTRO_CIRURGICO,ST_CENTRO_OBSTETRICO,ST_CENTRO_NEONATAL,ST_ATEND_HOSPITALAR,ST_SERVICO_APOIO,ST_ATEND_AMBULATORIAL,CO_MOTIVO_DESAB,CO_AMBULATORIAL_SUS
0,8486,3547800008486,35,354780,46522942000130,MUNICIPIO DE SANTO ANDRE,CENTRO MEDICO DE ESPECIALIDADES JOAQUIM TAVORA,,,M,,,M,MUNICIPAL,04,4,09030390,RUA JOAQUIM TAVORA,134,VILA ASSUNCAO,(11) 4435-1284,-23.6681857,-46.5248083,04,"ATENDIMENTO NOS TURNOS DA MANHA, TARDE E NOITE",,,1244,0.0,0.0,0.0,0.0,1.0,0.0,10,SIM
1,8494,3547800008494,35,354780,46522942000130,PREFEITURA MUNICIPAL DE SANTO ANDRE,UPA BANGU,,,M,,,M,MUNICIPAL,04,73,09210240,RUA AVARE,107,VILA SAO PEDRO,(11)49964646,-23.6411957,-46.5274164,06,"ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:INCLUI SABADOS, DOMINGOS E FERIADOS)",,,1244,0.0,0.0,0.0,0.0,1.0,1.0,,SIM
2,8508,3547800008508,35,354780,46522942000130,PREFEITURA MUNICIPAL DE SANTO ANDRE,UBS DR MOYSES FUCS,,,M,,,M,MUNICIPAL,03,2,09240200,RUA ALEXANDRETA,180,JARDIM SANTO ANTONIO,(11) 4435-1373,-23.6282563,-46.5223628,04,"ATENDIMENTO NOS TURNOS DA MANHA, TARDE E NOITE",,,1244,0.0,0.0,0.0,0.0,1.0,0.0,,SIM


## 9. Conferência do campo de identificação municipal

Confirma que o campo CO_IBGE foi localizado e mostra exemplos dos códigos.

In [11]:
coluna_codigo_cnes = "CO_IBGE"

print("Coluna identificadora encontrada:", coluna_codigo_cnes)
print("Colunas relacionadas a município/IBGE:")
print([col for col in cnes_abc.columns if "mun" in str(col).lower() or "ibge" in str(col).lower()])

Coluna identificadora encontrada: CO_IBGE
Colunas relacionadas a município/IBGE:
['CO_IBGE']


## 10. Padronização dos códigos municipais

Converte o CO_IBGE de seis dígitos e prepara a chave de relacionamento com o IBGE.

In [12]:
print("Tipo de dado:", cnes_abc["CO_IBGE"].dtype)
display(cnes_abc[["CO_IBGE"]].head(10))
print("Municípios distintos no recorte:", cnes_abc["CO_IBGE"].nunique())

Tipo de dado: object


,CO_IBGE
0,354780
1,354780
2,354780
3,354780
4,354780
5,354780
6,354780
7,354780
8,354780
9,354780


Municípios distintos no recorte: 7


## 11. Contagem de registros CNES por município

Conta os registros encontrados em cada município e verifica a cobertura territorial.

In [13]:
# O CNES utiliza o código municipal de seis dígitos (sem o dígito verificador do IBGE).
cnes_abc["CO_IBGE"] = (
    cnes_abc["CO_IBGE"].astype("string").str.strip().str.zfill(6)
)

municipios_abc["codigo_ibge"] = municipios_abc["codigo_ibge"].astype("string").str.zfill(7)
municipios_abc["codigo_cnes_6"] = municipios_abc["codigo_ibge"].str[:6]

cnes_abc = cnes_abc.merge(
    municipios_abc[["codigo_ibge", "codigo_cnes_6", "municipio"]],
    left_on="CO_IBGE", right_on="codigo_cnes_6", how="left", validate="many_to_one"
)

print("Registros associados a municípios:", cnes_abc["municipio"].notna().sum())
display(cnes_abc[["CO_IBGE", "codigo_ibge", "municipio"]].head())

Registros associados a municípios: 6290


,CO_IBGE,codigo_ibge,municipio
0,354780,3547809,Santo André
1,354780,3547809,Santo André
2,354780,3547809,Santo André
3,354780,3547809,Santo André
4,354780,3547809,Santo André


## 12. Integração dos dados e cálculo do indicador

Combina a contagem CNES com a população e calcula registros por 10 mil habitantes.

In [14]:
contagem_cnes = (
    cnes_abc.groupby(["codigo_ibge", "municipio"], as_index=False)
    .agg(quantidade_registros=("CO_CNES", "nunique"))
)

conferencia_municipios = municipios_abc.merge(
    contagem_cnes, on=["codigo_ibge", "municipio"], how="left"
)
conferencia_municipios["quantidade_registros"] = (
    conferencia_municipios["quantidade_registros"].fillna(0).astype(int)
)

display(conferencia_municipios)
print("Municípios com registros:", (conferencia_municipios["quantidade_registros"] > 0).sum(), "de 7")

,codigo_ibge,municipio,uf,codigo_cnes_6,quantidade_registros
0,3513801,Diadema,SP,351380,366
1,3529401,Mauá,SP,352940,535
2,3543303,Ribeirão Pires,SP,354330,170
3,3544103,Rio Grande da Serra,SP,354410,40
4,3547809,Santo André,SP,354780,1721
5,3548708,São Bernardo do Campo,SP,354870,1954
6,3548807,São Caetano do Sul,SP,354880,1504


Municípios com registros: 7 de 7


## 13. Exportação das bases tratadas

Salva os resultados intermediários para análises futuras e uso no Streamlit.

In [15]:
base_municipal = conferencia_municipios.merge(
    populacao_ibge[["codigo_ibge", "populacao_2022"]],
    on="codigo_ibge", how="left"
)

base_municipal["registros_cnes_por_10mil_hab"] = (
    base_municipal["quantidade_registros"] / base_municipal["populacao_2022"] * 10000
)

display(base_municipal)

# Exporta os resultados para a pasta de dados tratados.
base_municipal.to_csv(PASTA_TRATADOS / "base_municipal_saude_em_foco.csv", index=False, encoding="utf-8-sig")
cnes_abc.to_csv(PASTA_TRATADOS / "cnes_estabelecimentos_abc.csv", index=False, encoding="utf-8-sig")
print("Bases exportadas para:", PASTA_TRATADOS.resolve())

,codigo_ibge,municipio,uf,codigo_cnes_6,quantidade_registros,populacao_2022,registros_cnes_por_10mil_hab
0,3513801,Diadema,SP,351380,366,393237,9.307364
1,3529401,Mauá,SP,352940,535,418261,12.791056
2,3543303,Ribeirão Pires,SP,354330,170,115559,14.711100
3,3544103,Rio Grande da Serra,SP,354410,40,44170,9.055920
4,3547809,Santo André,SP,354780,1721,748919,22.979788
5,3548708,São Bernardo do Campo,SP,354870,1954,810729,24.101765
6,3548807,São Caetano do Sul,SP,354880,1504,165655,90.791102


Bases exportadas para: C:\Users\Caue Almiron\Documents\global_fiap\dados\tratados


## Observações metodológicas e próximos passos

- A contagem inicial inclui os registros CNES encontrados no território; ainda não aplica filtros de situação cadastral.
- A natureza pública/privada deve ser classificada após inspeção dos campos de esfera administrativa/natureza jurídica.
- `CO_AMBULATORIAL_SUS` não equivale à natureza pública do estabelecimento.
- Quantidade de estabelecimentos não mede capacidade, qualidade ou acesso efetivo.
- O CNES e o Censo 2022 podem ter períodos de referência diferentes; documente a data de referência do arquivo CNES.
- A análise atual é municipal e não sustenta conclusões por bairro.

**Próxima etapa:** validar os campos do CNES para classificar estabelecimentos e, só então, recalcular os indicadores com filtros documentados.

In [16]:
# 1. Visualizar todas as colunas disponíveis
print("Quantidade de colunas:", len(cnes_abc.columns))

for coluna in cnes_abc.columns:
    print(coluna)

Quantidade de colunas: 39
CO_CNES
CO_UNIDADE
CO_UF
CO_IBGE
NU_CNPJ_MANTENEDORA
NO_RAZAO_SOCIAL
NO_FANTASIA
CO_NATUREZA_ORGANIZACAO
DS_NATUREZA_ORGANIZACAO
TP_GESTAO
CO_NIVEL_HIERARQUIA
DS_NIVEL_HIERARQUIA
CO_ESFERA_ADMINISTRATIVA
DS_ESFERA_ADMINISTRATIVA
CO_ATIVIDADE
TP_UNIDADE
CO_CEP
NO_LOGRADOURO
NU_ENDERECO
NO_BAIRRO
NU_TELEFONE
NU_LATITUDE
NU_LONGITUDE
CO_TURNO_ATENDIMENTO
DS_TURNO_ATENDIMENTO
NU_CNPJ
NO_EMAIL
CO_NATUREZA_JUR
ST_CENTRO_CIRURGICO
ST_CENTRO_OBSTETRICO
ST_CENTRO_NEONATAL
ST_ATEND_HOSPITALAR
ST_SERVICO_APOIO
ST_ATEND_AMBULATORIAL
CO_MOTIVO_DESAB
CO_AMBULATORIAL_SUS
codigo_ibge
codigo_cnes_6
municipio


In [17]:
# 2. Verificar possíveis campos de identificação e classificação

colunas_interesse = [
    coluna for coluna in cnes_abc.columns
    if any(
        termo in coluna.upper()
        for termo in [
            "CNES",
            "ESFERA",
            "NATUREZA",
            "GESTAO",
            "SUS",
            "SITUACAO",
            "ATIV"
        ]
    )
]

print("Colunas relevantes encontradas:")

for coluna in colunas_interesse:
    print(coluna)

Colunas relevantes encontradas:
CO_CNES
CO_NATUREZA_ORGANIZACAO
DS_NATUREZA_ORGANIZACAO
TP_GESTAO
CO_ESFERA_ADMINISTRATIVA
DS_ESFERA_ADMINISTRATIVA
CO_ATIVIDADE
CO_NATUREZA_JUR
CO_AMBULATORIAL_SUS
codigo_cnes_6


In [18]:
# 3. Verificar a quantidade de códigos CNES únicos e duplicados

print("Total de registros:", len(cnes_abc))

print(
    "Estabelecimentos CNES únicos:",
    cnes_abc["CO_CNES"].nunique()
)

print(
    "Registros duplicados por código CNES:",
    cnes_abc.duplicated(subset=["CO_CNES"]).sum()
)

Total de registros: 6290
Estabelecimentos CNES únicos: 6290
Registros duplicados por código CNES: 0


In [19]:

# 1. Verificar os valores da esfera administrativa

print("ESFERA ADMINISTRATIVA")
display(
    cnes_abc["DS_ESFERA_ADMINISTRATIVA"]
    .value_counts(dropna=False)
    .to_frame("quantidade")
)

# 2. Verificar a natureza da organização

print("NATUREZA DA ORGANIZAÇÃO")
display(
    cnes_abc["DS_NATUREZA_ORGANIZACAO"]
    .value_counts(dropna=False)
    .to_frame("quantidade")
)

# 3. Verificar a natureza jurídica

print("NATUREZA JURÍDICA")
display(
    cnes_abc["CO_NATUREZA_JUR"]
    .value_counts(dropna=False)
    .to_frame("quantidade")
)

# 4. Verificar a atividade

print("ATIVIDADE")
display(
    cnes_abc["CO_ATIVIDADE"]
    .value_counts(dropna=False)
    .to_frame("quantidade")
)

# 5. Verificar atendimento ambulatorial pelo SUS

print("ATENDIMENTO AMBULATORIAL SUS")
display(
    cnes_abc["CO_AMBULATORIAL_SUS"]
    .value_counts(dropna=False)
    .to_frame("quantidade")
)

# 6. Verificar tipo de gestão

print("TIPO DE GESTÃO")
display(
    cnes_abc["TP_GESTAO"]
    .value_counts(dropna=False)
    .to_frame("quantidade")
)


ESFERA ADMINISTRATIVA


,quantidade
DS_ESFERA_ADMINISTRATIVA,
MUNICIPAL,6269
ESTADUAL,21


NATUREZA DA ORGANIZAÇÃO


,quantidade
DS_NATUREZA_ORGANIZACAO,
,6289
EMPRESA PRIVADA,1


NATUREZA JURÍDICA


,quantidade
CO_NATUREZA_JUR,
2062,2684
4000,1209
2240,963
1244,471
2135,305
2046,153
2054,144
3999,87
2000,60


... (2 outputs left out)

ATENDIMENTO AMBULATORIAL SUS


,quantidade
CO_AMBULATORIAL_SUS,
NAO,5886
SIM,404


TIPO DE GESTÃO


,quantidade
TP_GESTAO,
M,6269
E,21


In [20]:

# Distribuição da esfera administrativa por município

tabela_esfera = pd.crosstab(
    cnes_abc["municipio"],
    cnes_abc["DS_ESFERA_ADMINISTRATIVA"],
    margins=True
)

display(tabela_esfera)


DS_ESFERA_ADMINISTRATIVA,ESTADUAL,MUNICIPAL,All
municipio,,,
Diadema,6,360,366
Mauá,2,533,535
Ribeirão Pires,0,170,170
Rio Grande da Serra,0,40,40
Santo André,8,1713,1721
São Bernardo do Campo,2,1952,1954
São Caetano do Sul,3,1501,1504
All,21,6269,6290
